In [13]:
import torch
from datasets import load_dataset
import numpy as np
import numba
from tqdm.notebook import tqdm
from collections import Counter
import regex as re
import multiprocessing as mp
from functools import reduce
import operator
from functools import partial
import numpy as np
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import os
import random

from lib.tokenizers.raw_tokenizers import RegexBPE
from lib.model_layers.transformer import Transformer

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


In [14]:
class SimpleStoriesBPEDataset(Dataset):
    def __init__(self, hf_split, max_length=None, pad_id=None, end_id=None):
        self.tok = RegexBPE()
        self.tok.load_merge_dict("transformer/merge_dict.pickle")
        self.tok.load_vocab("transformer/vocab.pickle")
        self.ds = hf_split
        self.max_length = max_length
        if pad_id is None:
            try:
                self.pad_id = max(self.tok.vocab.keys()) + 1
            except:
                self.pad_id = 0
        else:
            self.pad_id = pad_id
        if end_id is None:
            try:
                self.end_id = max(self.tok.vocab.keys()) + 2
            except:
                self.end_id = 0
        else:
            self.end_id = end_id
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, i):
        ids = self.tok.encode(self.ds[i]["story"])
        if self.end_id is not None:
            ids = ids + [self.end_id]
        if self.max_length is not None:
            ids = ids[:self.max_length]
        return torch.tensor(ids, dtype=torch.long)



In [15]:
data = load_dataset("SimpleStories/SimpleStories")
dataset = SimpleStoriesBPEDataset(data["train"], max_length=500)
def pad_collate(batch):
    x = torch.nn.utils.rnn.pad_sequence(batch, batch_first=True, padding_value=dataset.pad_id)
    m = (x != dataset.pad_id).long()
    return {"input_ids": x, "attention_mask": m}


loader = DataLoader(dataset, batch_size=24, shuffle=True, collate_fn=pad_collate, num_workers=4)

In [16]:
torch.set_default_dtype(torch.bfloat16)
model = Transformer(8, max_context=500).to(dtype=torch.bfloat16, device=device)
model.embedding.weight.data = model.embedding.weight.data.to(torch.bfloat16)
model.embedding.padding_idx = dataset.pad_id
criterion = torch.nn.CrossEntropyLoss(ignore_index=dataset.pad_id)

In [17]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
epochs = 3
save_dir = "checkpoints"
os.makedirs(save_dir, exist_ok=True)
best_loss = float("inf")
global_step = 0
loss_steps, loss_values = [], []
prompt_text = "The dog"

def generate_sample(prompt, n_words=15, max_new_tokens=60):
    model.eval()
    with torch.no_grad():
        ids0 = dataset.tok.encode(prompt)
        if len(ids0) == 0:
            ids0 = [dataset.pad_id]
        ids = torch.tensor(ids0, dtype=torch.long, device=device).unsqueeze(0)
        for _ in range(max_new_tokens):
            logits = model(ids)
            next_id = int(logits[0, -1].argmax(-1).item())
            if dataset.end_id is not None and next_id == dataset.end_id:
                break
            ids = torch.cat([ids, torch.tensor([[next_id]], device=device)], dim=1)
            if len(dataset.tok.decode(ids[0].tolist()).split()) >= n_words:
                break
    model.train()
    return " ".join(dataset.tok.decode(ids[0].tolist()).split()[:n_words])


for epoch in range(1, epochs + 1):
    model.train()
    pbar = tqdm(loader, desc=f"epoch {epoch}/{epochs}")
    epoch_loss, n_batches = 0.0, 0
    for batch in pbar:
        if global_step % 10 == 0:
            try:
                s = generate_sample(prompt_text, 15, 60)
                pbar.write(s)
            except Exception as e:
                pbar.write(f"[gen err: {e}]")
        if global_step % 500 == 0:
            torch.save(
                {"model": model.state_dict(), "optimizer": optimizer.state_dict(), "step": global_step, "epoch": epoch},
                os.path.join(save_dir, f"ckpt_step_{global_step}.pt"),
            )
            torch.save(
                {"model": model.state_dict(), "optimizer": optimizer.state_dict(), "step": global_step, "epoch": epoch},
                os.path.join(save_dir, "ckpt_latest.pt"),
            )
            torch.save({"steps": loss_steps, "losses": loss_values}, os.path.join(save_dir, "loss_history.pt"))
        x = batch["input_ids"].to(device, non_blocking=True)
        logits = model(x[:, :-1])
        loss = criterion(logits.reshape(-1, logits.size(-1)), x[:, 1:].reshape(-1))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        global_step += 1
        epoch_loss += loss.item()
        n_batches += 1
        loss_steps.append(global_step)
        loss_values.append(loss.item())
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    avg_loss = epoch_loss / max(1, n_batches)
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(
            {"model": model.state_dict(), "optimizer": optimizer.state_dict(), "step": global_step, "epoch": epoch},
            os.path.join(save_dir, "ckpt_best.pt"),
        )
    torch.save({"steps": loss_steps, "losses": loss_values}, os.path.join(save_dir, "loss_history.pt"))

epoch 1/3:   0%|          | 0/88154 [00:00<?, ?it/s]

Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=83, pipe_handle=182)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=83, pipe_handle=196)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/samsutherland/miniconda3/envs/ai/lib/python3.13/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/Users/samsutherland/miniconda3/envs/ai/lib/python3.13/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_mai

RuntimeError: DataLoader worker (pid(s) 32792, 32794) exited unexpectedly